## Week 11: DBSCAN and HAC Clustering

This week I am going to be applying DBSCAN and HAC Clustering to my datasets to see how the data clusters together

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics import roc_curve, auc, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

### Dataset 1: Instacart Dataset

In [3]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [4]:
X = data1_transformed.drop(columns = ['ordered', 'user_id', 'product_id'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [5]:
dbscan = DBSCAN(eps = 0.5, min_samples = 5)

test_clusters = dbscan.fit_predict(X_test_scaled)

dbscan_profiles = X_test.copy()
dbscan_profiles['cluster'] = test_clusters

dbscan_profiles.groupby('cluster').mean()

,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
cluster,,,,,,,
-1,1.919793,3.299928,0.264500,0.071244,2.956815,3.299928,1.387070
0,1.098612,2.583413,0.179682,0.005643,2.583413,2.583413,0.000000
1,0.693147,2.404092,0.116162,0.004428,2.404092,2.404092,0.000000
2,0.693147,2.953851,0.073376,0.003638,2.133312,2.953851,2.166873
3,1.757028,3.418424,0.180210,0.005491,3.079412,3.418424,1.881428
...,...,...,...,...,...,...,...
128,3.330607,3.707399,0.518047,0.000523,3.610379,3.707399,1.517106
129,0.693147,4.251371,0.014411,0.044971,3.227460,4.251371,3.821192
130,0.693147,3.702523,0.025045,0.070374,3.343855,3.702523,2.536118


In [10]:
sample_idx = np.random.choice(len(X_test_scaled), size = min(2000, len(X_test_scaled)), replace = False)

X_hac = X_test_scaled[sample_idx]

hac = AgglomerativeClustering(n_clusters = 4, linkage = 'ward')

test_clusters = hac.fit_predict(X_hac)

hac_profiles = pd.DataFrame(X_hac, columns = X_test.columns)
hac_profiles['cluster'] = test_clusters

hac_profiles.groupby('cluster').mean()

,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
cluster,,,,,,,
0,2.525717,0.555841,1.393277,1.511844,0.969372,0.555841,-0.781957
1,-0.287714,-1.246918,0.840639,-0.086818,-0.889667,-1.246918,-0.771922
2,-0.309130,0.068429,-0.416201,-0.111362,-0.166197,0.068429,0.240114
3,0.133119,1.201793,-0.667567,-0.198068,1.111832,1.201793,0.715341


In [11]:
hac_profiles['target'] = y_test.iloc[sample_idx].values

hac_profiles.groupby('cluster')['target'].mean()

cluster
0    0.206061
1    0.098039
2    0.035242
3    0.026379
Name: target, dtype: float64

### Dataset 2: Amazon Reviews Dataset

In [12]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked', 'rating'])
y = data2_transformed['liked']

X.head()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,False,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,True,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,True,1,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,True,0,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,True,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [15]:
dbscan = DBSCAN(eps = 0.5, min_samples = 5)

test_clusters = dbscan.fit_predict(X_test_scaled)

dbscan_profiles = X_test.copy()
dbscan_profiles['cluster'] = test_clusters

dbscan_profiles.groupby('cluster').mean()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
cluster,,,,,,,,,,,,,,,,,,,,,
-1,0.849315,0.273973,0.315068,0.246575,0.157534,0.164384,0.150685,0.054795,0.068493,0.075342,...,0.013699,0.020548,0.020548,0.013699,0.013699,0.266099,1.69486,2.843919,0.84574,1.887281


In [18]:
pd.set_option('display.max_columns', None)

sample_idx = np.random.choice(len(X_test_scaled), size = min(2000, len(X_test_scaled)), replace = False)

X_hac = X_test_scaled[sample_idx]

hac = AgglomerativeClustering(n_clusters = 4, linkage = 'ward')

test_clusters = hac.fit_predict(X_hac)

hac_profiles = pd.DataFrame(X_hac, columns = X_test.columns)
hac_profiles['cluster'] = test_clusters

hac_profiles.groupby('cluster').mean()

,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,Breads & Bakery,Single-Serve Capsules & Pods,Tea,Cookies,Meat Snacks,"Soups, Stocks & Broths",Nuts & Seeds,Jerky,Juices,Ground Coffee,Breakfast Foods,Cereals,Granola,Fruit & Herbal Tea,Herbal,Bars,Roasted Coffee Beans,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,-0.137851,1.627882,-0.678233,-0.572078,1.371407,-0.443533,-0.421212,0.763590,-0.271163,-0.285450,0.905283,0.218878,-0.224410,-0.256307,-0.240772,-0.224410,-0.240772,0.459354,0.812860,-0.188311,-0.188311,-0.224410,0.532281,0.532281,-0.20702,0.373756,0.373756,-0.144841,-0.144841,-0.117851,-0.117851,-0.198357,0.191954,0.153654,0.149567,0.025471
1,0.056606,-0.492438,0.070516,0.335785,-0.432426,-0.150255,0.247233,-0.240772,0.159161,0.167547,-0.285450,-0.030336,0.131719,-0.211113,0.141323,0.131719,-0.240772,-0.144841,-0.256307,-0.188311,-0.188311,-0.224410,-0.167836,-0.167836,-0.20702,-0.117851,-0.117851,0.085016,-0.144841,0.069173,0.069173,0.124742,-0.104907,0.032622,-0.140394,0.015594
2,-0.696914,-0.614295,-0.678233,-0.572078,-0.432426,-0.443533,-0.421212,-0.240772,-0.271163,-0.285450,-0.285450,-0.256307,-0.224410,-0.256307,-0.240772,-0.224410,-0.240772,-0.144841,-0.256307,5.310367,5.310367,0.711699,-0.167836,-0.167836,-0.20702,-0.117851,-0.117851,-0.144841,4.084526,-0.117851,-0.117851,-0.528377,0.162183,-0.696945,0.290767,0.155041
3,0.221546,-0.614295,1.474420,-0.572078,-0.432426,2.254625,-0.421212,-0.240772,-0.271163,-0.285450,-0.285450,-0.256307,-0.224410,2.119621,-0.240772,-0.224410,2.270133,-0.144841,-0.256307,-0.188311,-0.188311,1.781538,-0.167836,-0.167836,1.95190,-0.117851,-0.117851,-0.144841,-0.144841,-0.117851,-0.117851,-0.135136,0.151581,-0.349599,0.444828,-0.221527


In [17]:
hac_profiles['target'] = y_test.iloc[sample_idx].values

hac_profiles.groupby('cluster')['target'].mean()

cluster
0    0.885714
1    0.815217
2    1.000000
3    0.785714
Name: target, dtype: float64

## Conclusion

For these methods, I had to opt for selecting the parameters as tuning was leading to the kernel crashing every time due to the memory it used up. Looking at the results, the DBScan results definitely were surprising due to the number of clusters it showed. As for the HAC, I did notice that the highest rated cluster has a positive high value for a lot of the features compared to the others in the first dataset. As for the second dataset, I couldn't identify any noticeable trends that made the third cluster stand out.